In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 accelerate peft wandb datasets huggingface_hub
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py
%env CUDA_VISIBLE_DEVICES=0

In [ ]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime

# Fetch secrets from Kaggle securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

# Log in to HuggingFace
login(hf_token, add_to_git_credential=True)

# Log in to Weights & Biases
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
PROJECT_NAME = "price"
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
# Constants
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct" 
HF_USER = "sairuthwik112"

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_full" 

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-full" 
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}" 
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}" 

# Hyper-parameters - overall
EPOCHS = 1 
BATCH_SIZE = 8 
MAX_SEQUENCE_LENGTH = 512 
GRADIENT_ACCUMULATION_STEPS = 4 

# Hyper-parameters - QLoRA
LORA_R = 64 
LORA_ALPHA = LORA_R * 2 
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] 
LORA_DROPOUT = 0.02 

# Hyper-parameters - training
LEARNING_RATE = 2e-4 
WARMUP_RATIO = 0.03 
LR_SCHEDULER_TYPE = 'cosine' 
WEIGHT_DECAY = 0.01 
OPTIMIZER = "paged_adamw_32bit" 

capability = torch.cuda.get_device_capability() 
use_bf16 = capability[0] >= 8 

# Tracking
LOG_STEPS = 50
SAVE_STEPS = 500
VAL_SIZE = 5000
print("loaded")

In [ ]:
dataset = load_dataset(DATASET_NAME) #[cite: 1]
train = dataset['train'] #[cite: 1]
val = dataset['val'].select(range(VAL_SIZE)) #[cite: 1]

wandb.init(project=PROJECT_NAME, name=RUN_NAME) #[cite: 1]

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
)

# Load the Tokenizer and the Model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": 0}, # <--- FORCES MODEL ONTO A SINGLE GPU FOR MAXIMUM SPEED
) 
base_model.generation_config.pad_token_id = tokenizer.pad_token_id 

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
from transformers import DataCollatorForLanguageModeling
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# 1. Clean Collator to strip away the breaking 'completion_mask' column
class QwenTextCollator:
    def __init__(self, tokenizer):
        self.base_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        
    def __call__(self, features):
        cleaned_features = []
        for feature in features:
            # ONLY keep the essential columns Qwen needs for training
            cleaned = {k: v for k, v in feature.items() if k in ["input_ids", "attention_mask", "labels"]}
            cleaned_features.append(cleaned)
            
        # Safely pad the clean batch without crashing
        return self.base_collator(cleaned_features)

# Initialize the new clean collator
collator = QwenTextCollator(tokenizer)

# 2. LoRA Parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# 3. Training parameters
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,          
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=1.0,
    max_steps=2500, # <--- STOPS TRAINING BEFORE KAGGLE KILLS THE SESSION
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="prompt",
    remove_unused_columns=True
)

# 4. Trainer Initialization
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
    processing_class=tokenizer, 
    data_collator=collator 
)

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

# Close the Weights & Biases run properly
wandb.finish()

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Re-authenticate securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# 2. Define the path to your final successful checkpoint directory
# (Double check your right-hand file panel to ensure this matches your output folder name!)
CHECKPOINT_PATH = "./price-prediction-YOUR_RUN_NAME/checkpoint-1500" 

print("Loading your trained weights...")
# Load the raw base model config first
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map={"": 0},
    trust_remote_code=True
)

# Load your custom weights on top
final_model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)

print("Pushing completed weights to Hugging Face...")
# Push to your profile
final_model.push_to_hub("price-prediction-qwen-final", private=True)

print("🎉 Success! Your fine-tuned model is securely uploaded to Hugging Face!")

In [ ]:
import os
import glob
from peft import PeftModel
from transformers import AutoModelForCausalLM
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Re-authenticate securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# 2. Automatically find your checkpoint-1500 directory on Kaggle
checkpoint_lookups = glob.glob("./**/checkpoint-1500", recursive=True)

if not checkpoint_lookups:
    # Fallback: check inside the standard Kaggle output working directory
    checkpoint_lookups = glob.glob("/kaggle/working/**/checkpoint-1500", recursive=True)

if not checkpoint_lookups:
    raise FileNotFoundError(
        "Could not automatically locate 'checkpoint-1500'. "
        "Please look at your right-hand file panel under /kaggle/working/ and check the folder name."
    )

CHECKPOINT_PATH = checkpoint_lookups[0]
print(f"🎯 Found your trained weights automatically at: {CHECKPOINT_PATH}")

print("Loading base Qwen model architecture...")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map={"": 0},
    trust_remote_code=True
)

print("Mounting your custom trained weights...")
final_model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)

print("Pushing completed weights to Hugging Face...")
# This pushes to your personal Hugging Face repository
final_model.push_to_hub("price-prediction-qwen-final", private=True)

print("🎉 Success! Your fine-tuned model is securely uploaded to Hugging Face!")

In [6]:
!pip install -q --upgrade torchao

In [7]:
import os
import glob
from peft import PeftModel
from transformers import AutoModelForCausalLM
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Re-authenticate securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# 2. Automatically find your checkpoint-1500 directory on Kaggle
checkpoint_lookups = glob.glob("./**/checkpoint-1500", recursive=True)

if not checkpoint_lookups:
    # Fallback: check inside the standard Kaggle output working directory
    checkpoint_lookups = glob.glob("/kaggle/working/**/checkpoint-1500", recursive=True)

if not checkpoint_lookups:
    raise FileNotFoundError(
        "Could not automatically locate 'checkpoint-1500'. "
        "Please look at your right-hand file panel under /kaggle/working/ and check the folder name."
    )

CHECKPOINT_PATH = checkpoint_lookups[0]
print(f"🎯 Found your trained weights automatically at: {CHECKPOINT_PATH}")

print("Loading base Qwen model architecture...")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map={"": 0},
    trust_remote_code=True
)

print("Mounting your custom trained weights...")
final_model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)

print("Pushing completed weights to Hugging Face...")
# This pushes to your personal Hugging Face repository
final_model.push_to_hub("price-prediction-qwen-final", private=True)

print("🎉 Success! Your fine-tuned model is securely uploaded to Hugging Face!")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


🎯 Found your trained weights automatically at: ./price-2026-07-15_15.18.55-full/checkpoint-1500
Loading base Qwen model architecture...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Mounting your custom trained weights...
Pushing completed weights to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 Success! Your fine-tuned model is securely uploaded to Hugging Face!
